# 01 — Construcción del dataset maestro

**Trabajo Final de Maestría — UNIR**  
**Equipo 1B**

Este notebook implementa la etapa de construcción del dataset maestro utilizado en el proyecto de predicción mensual de remesas familiares hacia El Salvador para el período 1991–2025.

El proceso integra información de múltiples fuentes, incluyendo datos de remesas, variables macroeconómicas de Estados Unidos y El Salvador, y registros de deportaciones. Posteriormente, los datos se homologan temporalmente e integran en una estructura mensual común, incorporando las variables derivadas necesarias para las etapas posteriores de análisis exploratorio y modelado.

**Salida principal:** `Dataset_Maestro_Final.csv`

**Nota de reproducibilidad:** algunas fuentes externas consultadas durante la construcción del dataset pueden actualizar o revisar sus series históricas. Por esta razón, una nueva ejecución del pipeline puede producir diferencias puntuales respecto a la versión del dataset utilizada originalmente para los resultados reportados en el informe.

In [2]:
import pandas as pd

# ============================================================
# 1. CREACIÓN DEL ÍNDICE TEMPORAL MAESTRO
# ============================================================

# Crear un rango de fechas mensual desde enero de 1991
# hasta diciembre de 2025.
fechas = pd.date_range(
    start='1991-01-01',
    end='2025-12-01',
    freq='MS'
)

# Crear el DataFrame maestro utilizando la fecha como índice.
df_maestro = pd.DataFrame(index=fechas)
df_maestro.index.name = 'Fecha'

# Verificación de cobertura temporal.
print("Inicio del dataset:")
print(df_maestro.head(3))

print("\nFinal del dataset:")
print(df_maestro.tail(3))

print(f"\nTotal de meses: {len(df_maestro)}")

# ============================================================
# CONFIGURACIÓN DE RUTAS DEL PROYECTO
# ============================================================

from pathlib import Path

# Detectar la raíz del proyecto.
# Funciona si el notebook se ejecuta desde la carpeta raíz
# o desde la carpeta notebooks/.
CWD = Path.cwd()

if (CWD / 'data').exists() and (CWD / 'notebooks').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'data').exists() and (CWD.parent / 'notebooks').exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto. "
        "Se esperaban las carpetas 'data' y 'notebooks'."
    )

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs_modelado'

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos: {DATA_DIR}")

Inicio del dataset:
Empty DataFrame
Columns: []
Index: [1991-01-01 00:00:00, 1991-02-01 00:00:00, 1991-03-01 00:00:00]

Final del dataset:
Empty DataFrame
Columns: []
Index: [2025-10-01 00:00:00, 2025-11-01 00:00:00, 2025-12-01 00:00:00]

Total de meses: 420
Raíz del proyecto: C:\Users\kmabe\TFM
Directorio de datos: C:\Users\kmabe\TFM\data


In [3]:
# ============================================================
# 2. CARGA DE DATOS DE REMESAS — BCR
# ============================================================

# Cargar el archivo fuente de remesas descargado del
# Banco Central de Reserva de El Salvador (BCR).
df_remesas_crudo = pd.read_csv(
    DATA_DIR / 'remesas_bcr.csv'
)

# Inspección inicial de la estructura del archivo.
print("Vista previa de los datos de remesas del BCR:")
print(df_remesas_crudo.head())

print(f"\nDimensiones del archivo: {df_remesas_crudo.shape}")

Vista previa de los datos de remesas del BCR:
            Ingresos mensuales de remesas familiares Unnamed: 1 Unnamed: 2  \
0               Datos actualizados hasta: Marzo 2026        NaN        NaN   
1                                    Millones de US$        NaN        NaN   
2                                           Concepto    1991-01    1991-02   
3           Ingresos mensuales de remesas familiares       63.1       58.4   
4  Notas Relevantes\n (p) Cifras preliminares\n F...        NaN        NaN   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        NaN        NaN        NaN        NaN        NaN        NaN   
1        NaN        NaN        NaN        NaN        NaN        NaN   
2    1991-03    1991-04    1991-05    1991-06    1991-07    1991-08   
3       67.6       77.8       77.4       67.8         70       53.5   
4        NaN        NaN        NaN        NaN        NaN        NaN   

  Unnamed: 9  ... Unnamed: 411 Unnamed: 412 Unnamed: 413 U

In [4]:
# ============================================================
# 3. TRANSFORMACIÓN E INTEGRACIÓN DE REMESAS
# ============================================================

# Extraer las fechas y los valores de remesas desde la
# estructura original del archivo fuente del BCR.
fechas_crudas = df_remesas_crudo.iloc[2, 1:].values
valores_crudos = df_remesas_crudo.iloc[3, 1:].values

# Limpiar las etiquetas de fecha eliminando la marca "(p)"
# correspondiente a valores preliminares y espacios adicionales.
fechas_limpias = [
    str(fecha).replace('(p)', '').strip()
    for fecha in fechas_crudas
]

# Construir un DataFrame temporal con estructura mensual.
df_temp = pd.DataFrame({
    'Fecha': pd.to_datetime(
        fechas_limpias,
        format='%Y-%m'
    ),
    'Remesas_Millones_USD': valores_crudos
})

# Convertir los valores de remesas a formato numérico.
# Los valores no convertibles se transforman en NaN.
df_temp['Remesas_Millones_USD'] = pd.to_numeric(
    df_temp['Remesas_Millones_USD'],
    errors='coerce'
)

# Establecer la fecha como índice temporal.
df_temp.set_index('Fecha', inplace=True)

# Integrar la serie mensual de remesas en el dataset maestro.
df_maestro['Remesas_Millones_USD'] = (
    df_temp['Remesas_Millones_USD']
)

# Verificación de la integración.
print("Dataset maestro actualizado — primeros 5 meses:")
print(df_maestro.head())

print("\nDataset maestro actualizado — últimas observaciones disponibles:")
print(df_maestro.dropna().tail())

nulos_remesas = (
    df_maestro['Remesas_Millones_USD']
    .isnull()
    .sum()
)

print(f"\nValores nulos en Remesas_Millones_USD: {nulos_remesas}")

Dataset maestro actualizado — primeros 5 meses:
            Remesas_Millones_USD
Fecha                           
1991-01-01                  63.1
1991-02-01                  58.4
1991-03-01                  67.6
1991-04-01                  77.8
1991-05-01                  77.4

Dataset maestro actualizado — últimas observaciones disponibles:
            Remesas_Millones_USD
Fecha                           
2025-08-01                824.96
2025-09-01                819.74
2025-10-01                852.99
2025-11-01                814.82
2025-12-01                961.12

Valores nulos en Remesas_Millones_USD: 0


In [5]:
# ============================================================
# 4. EXTRACCIÓN E INTEGRACIÓN DE VARIABLES DE FRED
# ============================================================

from fredapi import Fred
import os

# Obtener la clave de acceso desde una variable de entorno.
FRED_API_KEY = os.getenv("FRED_API_KEY")

if not FRED_API_KEY:
    raise ValueError(
        "No se encontró la variable de entorno FRED_API_KEY. "
        "Configúrala antes de ejecutar esta sección."
    )

# Inicializar el cliente de FRED.
fred = Fred(api_key=FRED_API_KEY)

# Definir las series macroeconómicas de Estados Unidos.
series_fred = {
    'Desempleo_Hispano_US': 'LNS14000009',
    'Desempleo_General_US': 'UNRATE',
    'Inflacion_CPI_US': 'CPIAUCSL',
    'Salario_Promedio_US': 'AHETPI',
    'PIB_US_Trimestral': 'GDP'
}

print("Extrayendo datos de FRED... (esto puede tomar unos segundos)")

# Descargar las series y almacenarlas en un DataFrame temporal.
df_fred = pd.DataFrame()

for nombre, codigo in series_fred.items():
    df_fred[nombre] = fred.get_series(codigo)

# Nombrar el índice temporal para mantener consistencia
# con el dataset maestro.
df_fred.index.name = 'Fecha'

# Eliminar columnas FRED existentes en caso de una
# ejecución previa de la celda.
for columna in df_fred.columns:
    if columna in df_maestro.columns:
        df_maestro = df_maestro.drop(columns=[columna])

# Integrar las variables de Estados Unidos mediante
# una unión temporal por fecha.
df_maestro = df_maestro.join(
    df_fred,
    how='left'
)

# Verificación de la integración.
print("\nValores nulos por variable:")
print(df_maestro.isnull().sum())

print("\nDataset maestro con variables de Estados Unidos:")
print(df_maestro.head())

Extrayendo datos de FRED... (esto puede tomar unos segundos)

Valores nulos por variable:
Remesas_Millones_USD      0
Desempleo_Hispano_US      1
Desempleo_General_US      1
Inflacion_CPI_US          1
Salario_Promedio_US       0
PIB_US_Trimestral       280
dtype: int64

Dataset maestro con variables de Estados Unidos:
            Remesas_Millones_USD  Desempleo_Hispano_US  Desempleo_General_US  \
Fecha                                                                          
1991-01-01                  63.1                   9.1                   6.4   
1991-02-01                  58.4                   9.2                   6.6   
1991-03-01                  67.6                   9.8                   6.8   
1991-04-01                  77.8                   9.8                   6.7   
1991-05-01                  77.4                  10.0                   6.9   

            Inflacion_CPI_US  Salario_Promedio_US  PIB_US_Trimestral  
Fecha                                          

In [6]:
# ============================================================
# 5. EXTRACCIÓN E INTEGRACIÓN DE VARIABLES DEL BANCO MUNDIAL
# ============================================================

import wbgapi as wb

print("Extrayendo datos del Banco Mundial... (esto puede tomar unos segundos)")

# Extraer indicadores anuales para El Salvador (código SLV):
# - NY.GDP.MKTP.CD: PIB nominal en USD corrientes.
# - FP.CPI.TOTL.ZG: inflación anual medida por el índice
#   de precios al consumidor.
pib_slv = wb.data.DataFrame(
    'NY.GDP.MKTP.CD',
    'SLV',
    time=range(1991, 2026)
).T

inf_slv = wb.data.DataFrame(
    'FP.CPI.TOTL.ZG',
    'SLV',
    time=range(1991, 2026)
).T

# Consolidar ambos indicadores en un DataFrame temporal.
df_bm = pd.concat(
    [pib_slv, inf_slv],
    axis=1
)

df_bm.columns = [
    'PIB_El_Salvador_Anual',
    'Inflacion_El_Salvador_Anual'
]

# Convertir el índice anual del Banco Mundial
# a un índice temporal compatible con el dataset maestro.
df_bm.index = (
    df_bm.index
    .str.replace('YR', '', regex=False)
    + '-01-01'
)

df_bm.index = pd.to_datetime(df_bm.index)
df_bm.index.name = 'Fecha'

# Eliminar columnas existentes en caso de una
# ejecución previa de la celda.
for columna in df_bm.columns:
    if columna in df_maestro.columns:
        df_maestro = df_maestro.drop(columns=[columna])

# Integrar los indicadores mediante una unión temporal por fecha.
df_maestro = df_maestro.join(
    df_bm,
    how='left'
)

# Verificación de la integración.
print("\nDataset maestro con variables del Banco Mundial:")
print(
    df_maestro[
        [
            'Remesas_Millones_USD',
            'PIB_El_Salvador_Anual',
            'Inflacion_El_Salvador_Anual'
        ]
    ].head(13)
)

Extrayendo datos del Banco Mundial... (esto puede tomar unos segundos)

Dataset maestro con variables del Banco Mundial:
            Remesas_Millones_USD  PIB_El_Salvador_Anual  \
Fecha                                                     
1991-01-01                  63.1           5.252342e+09   
1991-02-01                  58.4                    NaN   
1991-03-01                  67.6                    NaN   
1991-04-01                  77.8                    NaN   
1991-05-01                  77.4                    NaN   
1991-06-01                  67.8                    NaN   
1991-07-01                  70.0                    NaN   
1991-08-01                  53.5                    NaN   
1991-09-01                  53.1                    NaN   
1991-10-01                  64.0                    NaN   
1991-11-01                  64.3                    NaN   
1991-12-01                  73.1                    NaN   
1992-01-01                  65.0           5.813399e+

In [7]:
# ============================================================
# 6. FUNCIONES AUXILIARES PARA EXTRACCIÓN DE DEPORTACIONES — DHS
# ============================================================
#
# Los archivos históricos del DHS presentan estructuras distintas
# según el período. Las funciones siguientes localizan las hojas
# relevantes y extraen los registros correspondientes a El Salvador
# para formatos antiguos y modernos.
# ============================================================

# Localizar la hoja que contiene registros de El Salvador
# en archivos históricos.
def encontrar_hoja_el_salvador(archivo):

    xls = pd.ExcelFile(archivo)

    for hoja in xls.sheet_names:
        df = pd.read_excel(
            archivo,
            sheet_name=hoja,
            header=None
        )

        contiene = (
            df.astype(str)
              .apply(
                  lambda c: c.str.contains(
                      'El Salvador',
                      case=False,
                      na=False
                  )
              )
        )

        if contiene.any().any():
            return hoja

    return None

# Localizar la hoja que contiene la tabla de deportaciones
# en archivos DHS con estructura moderna.
def encontrar_hoja_deportaciones(archivo):

    xls = pd.ExcelFile(archivo)

    for hoja in xls.sheet_names:
        df = pd.read_excel(
            archivo,
            sheet_name=hoja,
            header=None
        )

        texto = (
            df.astype(str)
              .apply(lambda c: c.str.lower())
        )

        tiene_salvador = texto.apply(
            lambda c: c.str.contains(
                'el salvador',
                na=False
            )
        ).any().any()

        tiene_total = texto.apply(
            lambda c: c.str.contains(
                'total',
                na=False
            )
        ).any().any()

        tiene_criminal = texto.apply(
            lambda c: c.str.contains(
                'criminal',
                na=False
            )
        ).any().any()

        if (
            tiene_salvador
            and tiene_total
            and tiene_criminal
        ):
            return hoja

    return None

# Extraer la serie anual desde el formato histórico 1992–1996.
def extraer_formato_antiguo(archivo):

    hoja = encontrar_hoja_el_salvador(archivo)

    df = pd.read_excel(
        archivo,
        sheet_name=hoja,
        header=None
    )

    fila_idx = (
        df.astype(str)
          .apply(
              lambda c: c.str.contains(
                  'El Salvador',
                  case=False,
                  na=False
              )
          )
          .any(axis=1)
          .idxmax()
    )

    fila = df.loc[fila_idx]

    encabezados = df.loc[3]

    serie = {}

    for col in range(1, len(df.columns)):

        anio = encabezados[col]

        if pd.notna(anio):

            serie[int(anio)] = pd.to_numeric(
                fila[col],
                errors='coerce'
            )

    return serie


# Extraer series anuales desde archivos DHS con estructura moderna.
def extraer_formato_moderno(archivo):

    hoja = encontrar_hoja_deportaciones(archivo)

    df = pd.read_excel(
        archivo,
        sheet_name=hoja,
        header=None
    )

    fila_idx = (
        df.astype(str)
          .apply(
              lambda c: c.str.contains(
                  'El Salvador',
                  case=False,
                  na=False
              )
          )
          .any(axis=1)
          .idxmax()
    )

    fila = df.loc[fila_idx]

    # detectar fila de años
    fila_anios = None

    for i in range(min(15, len(df))):

        valores = pd.to_numeric(
            df.iloc[i],
            errors='coerce'
        )

        anios = valores[
            (valores >= 1900)
            & (valores <= 2100)
        ]

        if len(anios) >= 3:

            fila_anios = i
            break

    # detectar fila Total
    fila_tipo = None

    for i in range(
        fila_anios,
        fila_anios + 5
    ):

        texto = (
            df.iloc[i]
              .astype(str)
              .str.lower()
        )

        if texto.str.contains('total').any():

            fila_tipo = i
            break

    serie = {}

    for col in range(len(df.columns)):

        anio = pd.to_numeric(
            df.iloc[fila_anios, col],
            errors='coerce'
        )

        tipo = str(
            df.iloc[fila_tipo, col]
        ).strip().lower()

        if (
            pd.notna(anio)
            and tipo == 'total'
        ):

            serie[int(anio)] = pd.to_numeric(
                fila[col],
                errors='coerce'
            )

    return serie

In [8]:
# ============================================================
# 7. CONSOLIDACIÓN DE SERIES HISTÓRICAS DE DEPORTACIONES — DHS
# ============================================================

serie_deportaciones = {}

# Procesar el archivo histórico con estructura antigua.
serie_deportaciones.update(
    extraer_formato_antiguo(
        DATA_DIR / 'dhs_1992_1996.xls'
    )
)

# Procesar los archivos con estructura moderna en orden cronológico.
# En años superpuestos, se conserva el valor procedente del archivo
# procesado posteriormente, priorizando la actualización más reciente
# disponible dentro del conjunto de archivos DHS utilizado.
archivos_dhs_modernos = [
    DATA_DIR / 'dhs_1993_2000.xls',
    DATA_DIR / 'dhs_2001_2010.xls',
    DATA_DIR / 'dhs_2011_2020.xlsx',
    DATA_DIR / 'dhs_2013_2022.xlsx'
]

for archivo in archivos_dhs_modernos:
    serie_deportaciones.update(
        extraer_formato_moderno(
            archivo
        )
    )

# Construir un DataFrame anual a partir de la serie consolidada.
df_deportaciones = pd.DataFrame.from_dict(
    serie_deportaciones,
    orient='index',
    columns=['Deportaciones_Total']
)

df_deportaciones.index.name = 'Anio'

# Ordenar cronológicamente la serie.
df_deportaciones = (
    df_deportaciones
    .sort_index()
)

# Verificación de la serie consolidada.
print("Serie anual consolidada de deportaciones:")
print(df_deportaciones)

Serie anual consolidada de deportaciones:
      Deportaciones_Total
Anio                     
1992                 1962
1993                 2117
1994                 1900
1995                 1932
1996                 2493
1997                 3900
1998                 5339
1999                 3972
2000                 4507
2001                 3928
2002                 4066
2003                 5561
2004                 7269
2005                 8305
2006                11050
2007                20045
2008                20050
2009                20849
2010                19809
2011                17945
2012                18910
2013                21132
2014                26678
2015                21823
2016                20346
2017                18339
2018                14931
2019                18038
2020                12167
2021                 2790
2022                 7325


In [9]:
# ============================================================
# 8. CONTROL DE CALIDAD DE LA SERIE DE DEPORTACIONES
# ============================================================

print("=== CONTROL DE CALIDAD — DHS ===")

print(f"\nPrimer año disponible: {df_deportaciones.index.min()}")
print(f"Último año disponible: {df_deportaciones.index.max()}")
print(f"Cantidad de años: {len(df_deportaciones)}")

print("\nValores nulos:")
print(df_deportaciones.isnull().sum())

print("\nÚltimos registros:")
print(df_deportaciones.tail())

# Verificar continuidad anual de la serie.
anios_esperados = set(range(1992, 2023))
anios_disponibles = set(df_deportaciones.index.astype(int))
anios_faltantes = sorted(anios_esperados - anios_disponibles)

print("\nAños faltantes en la serie:")
print(anios_faltantes if anios_faltantes else "Ninguno")

=== CONTROL DE CALIDAD — DHS ===

Primer año disponible: 1992
Último año disponible: 2022
Cantidad de años: 31

Valores nulos:
Deportaciones_Total    0
dtype: int64

Últimos registros:
      Deportaciones_Total
Anio                     
2018                14931
2019                18038
2020                12167
2021                 2790
2022                 7325

Años faltantes en la serie:
Ninguno


In [10]:
# ============================================================
# 9. CONVERSIÓN DE LA SERIE DHS A FRECUENCIA MENSUAL
# ============================================================

# Crear una copia del dataset integrado hasta esta etapa.
df_maestro_final = df_maestro.copy()

# Convertir el índice anual de deportaciones a fechas.
df_deportaciones_mensual = df_deportaciones.copy()

df_deportaciones_mensual.index = pd.to_datetime(
    df_deportaciones_mensual.index.astype(str) + '-01-01'
)

df_deportaciones_mensual.index.name = 'Fecha'

# Crear un calendario mensual para el período cubierto
# por la serie consolidada del DHS: 1992–2022.
indice_mensual = pd.date_range(
    start='1992-01-01',
    end='2022-12-01',
    freq='MS'
)

# Reindexar la serie anual sobre el calendario mensual.
df_deportaciones_mensual = (
    df_deportaciones_mensual
    .reindex(indice_mensual)
)

# Construir una aproximación mensual de la serie mediante
# interpolación lineal entre los valores anuales disponibles.
#
# Esta transformación genera una serie mensual suavizada para
# su integración con el resto de variables del dataset.
# Los períodos sin cobertura DHS se conservan inicialmente
# como valores faltantes y se tratan posteriormente en la
# etapa de preparación para el modelado.
df_deportaciones_mensual['Deportaciones_Total'] = (
    df_deportaciones_mensual['Deportaciones_Total']
    .interpolate(method='linear')
)

# Eliminar la variable si existe por una ejecución previa.
if 'Deportaciones_Total' in df_maestro_final.columns:
    df_maestro_final = df_maestro_final.drop(
        columns=['Deportaciones_Total']
    )

# Integrar la serie mensual interpolada al dataset maestro.
df_maestro_final = df_maestro_final.join(
    df_deportaciones_mensual,
    how='left'
)

# Control de calidad.
print("Dimensiones del dataset:")
print(df_maestro_final.shape)

print("\nValores nulos en Deportaciones_Total:")
print(
    df_maestro_final[
        ['Deportaciones_Total']
    ].isnull().sum()
)

print("\nPrimeros registros:")
print(
    df_maestro_final[
        ['Deportaciones_Total']
    ].head(15)
)

print("\nÚltimos registros:")
print(
    df_maestro_final[
        ['Deportaciones_Total']
    ].tail(15)
)

Dimensiones del dataset:
(420, 9)

Valores nulos en Deportaciones_Total:
Deportaciones_Total    48
dtype: int64

Primeros registros:
            Deportaciones_Total
Fecha                          
1991-01-01                  NaN
1991-02-01                  NaN
1991-03-01                  NaN
1991-04-01                  NaN
1991-05-01                  NaN
1991-06-01                  NaN
1991-07-01                  NaN
1991-08-01                  NaN
1991-09-01                  NaN
1991-10-01                  NaN
1991-11-01                  NaN
1991-12-01                  NaN
1992-01-01          1962.000000
1992-02-01          1974.916667
1992-03-01          1987.833333

Últimos registros:
            Deportaciones_Total
Fecha                          
2024-10-01                  NaN
2024-11-01                  NaN
2024-12-01                  NaN
2025-01-01                  NaN
2025-02-01                  NaN
2025-03-01                  NaN
2025-04-01                  NaN
2025-05-01     

In [11]:
# ============================================================
# 10. TRATAMIENTO FINAL DE VALORES FALTANTES
# ============================================================

# Variables económicas con frecuencia original trimestral o anual.
columnas_economicas = [
    'PIB_US_Trimestral',
    'PIB_El_Salvador_Anual',
    'Inflacion_El_Salvador_Anual'
]

# Variables mensuales provenientes de FRED.
columnas_fred = [
    'Desempleo_Hispano_US',
    'Desempleo_General_US',
    'Inflacion_CPI_US'
]

# Propagar el último valor disponible hacia adelante y,
# cuando sea necesario al inicio de la serie, utilizar el
# primer valor posterior disponible.
#
# Esta estrategia conserva el valor observado hasta la
# siguiente actualización de las series trimestrales o anuales.
df_maestro_final[columnas_economicas] = (
    df_maestro_final[columnas_economicas]
    .ffill()
    .bfill()
)

# Completar faltantes puntuales de las series mensuales FRED
# mediante propagación hacia adelante y hacia atrás.
df_maestro_final[columnas_fred] = (
    df_maestro_final[columnas_fred]
    .ffill()
    .bfill()
)

# La variable Deportaciones_Total conserva valores faltantes
# fuera del período cubierto por la serie DHS (1991 y 2023–2025).
# Estos valores no se imputan en el dataset maestro para preservar
# la trazabilidad de la fuente y se tratan posteriormente en el
# notebook de modelado mediante supuestos explícitos.

# Diagnóstico posterior al tratamiento.
print("=== DIAGNÓSTICO DE VALORES FALTANTES ===")

print("\nNulos por variable:")
print(df_maestro_final.isnull().sum())

print(f"\nTotal de registros: {len(df_maestro_final)}")

print("\nPeríodo:")
print(
    f"{df_maestro_final.index.min().date()} "
    f"→ {df_maestro_final.index.max().date()}"
)

=== DIAGNÓSTICO DE VALORES FALTANTES ===

Nulos por variable:
Remesas_Millones_USD            0
Desempleo_Hispano_US            0
Desempleo_General_US            0
Inflacion_CPI_US                0
Salario_Promedio_US             0
PIB_US_Trimestral               0
PIB_El_Salvador_Anual           0
Inflacion_El_Salvador_Anual     0
Deportaciones_Total            48
dtype: int64

Total de registros: 420

Período:
1991-01-01 → 2025-12-01


In [12]:
# ============================================================
# 11. CREACIÓN DE VARIABLES REZAGADAS DE REMESAS
# ============================================================

# Crear rezagos de 1, 2, 3 y 12 meses de la variable objetivo.
# Estas variables permiten incorporar información histórica
# reciente y estacional de la serie de remesas.
for lag in [1, 2, 3, 12]:
    df_maestro_final[f'Remesas_Lag{lag}'] = (
        df_maestro_final['Remesas_Millones_USD']
        .shift(lag)
    )

# Verificación de las variables rezagadas.
columnas_rezagos = [
    'Remesas_Millones_USD',
    'Remesas_Lag1',
    'Remesas_Lag2',
    'Remesas_Lag3',
    'Remesas_Lag12'
]

print("Variables rezagadas de remesas:")
print(
    df_maestro_final[
        columnas_rezagos
    ].head(15)
)

Variables rezagadas de remesas:
            Remesas_Millones_USD  Remesas_Lag1  Remesas_Lag2  Remesas_Lag3  \
Fecha                                                                        
1991-01-01                  63.1           NaN           NaN           NaN   
1991-02-01                  58.4          63.1           NaN           NaN   
1991-03-01                  67.6          58.4          63.1           NaN   
1991-04-01                  77.8          67.6          58.4          63.1   
1991-05-01                  77.4          77.8          67.6          58.4   
1991-06-01                  67.8          77.4          77.8          67.6   
1991-07-01                  70.0          67.8          77.4          77.8   
1991-08-01                  53.5          70.0          67.8          77.4   
1991-09-01                  53.1          53.5          70.0          67.8   
1991-10-01                  64.0          53.1          53.5          70.0   
1991-11-01                  64.3

In [13]:
# ============================================================
# 12. CREACIÓN DE VARIABLES DUMMY
# ============================================================

# ------------------------------------------------------------
# 12.1. Dummy de crisis económica
# ------------------------------------------------------------

# Inicializar la variable en cero.
df_maestro_final['Dummy_Crisis'] = 0

# Crisis de 2001.
df_maestro_final.loc[
    (df_maestro_final.index >= '2001-01-01') &
    (df_maestro_final.index <= '2001-12-01'),
    'Dummy_Crisis'
] = 1

# Crisis financiera global de 2008–2009.
df_maestro_final.loc[
    (df_maestro_final.index >= '2008-01-01') &
    (df_maestro_final.index <= '2009-12-01'),
    'Dummy_Crisis'
] = 1

# Choque económico asociado a la pandemia en 2020.
df_maestro_final.loc[
    (df_maestro_final.index >= '2020-01-01') &
    (df_maestro_final.index <= '2020-12-01'),
    'Dummy_Crisis'
] = 1


# ------------------------------------------------------------
# 12.2. Dummy del período COVID-19
# ------------------------------------------------------------

# Identificar el período 2020–2022 definido en el proyecto
# para representar los efectos asociados al contexto pandémico.
df_maestro_final['Dummy_Covid'] = 0

df_maestro_final.loc[
    (df_maestro_final.index >= '2020-01-01') &
    (df_maestro_final.index <= '2022-12-01'),
    'Dummy_Covid'
] = 1


# ------------------------------------------------------------
# 12.3. Verificación
# ------------------------------------------------------------

print("Variables disponibles:")
print(df_maestro_final.columns.tolist())

print("\nDistribución de las variables dummy:")
print(
    df_maestro_final[
        ['Dummy_Crisis', 'Dummy_Covid']
    ].sum()
)

print("\nÚltimos 50 registros:")
print(
    df_maestro_final[
        ['Dummy_Crisis', 'Dummy_Covid']
    ].tail(50)
)

Variables disponibles:
['Remesas_Millones_USD', 'Desempleo_Hispano_US', 'Desempleo_General_US', 'Inflacion_CPI_US', 'Salario_Promedio_US', 'PIB_US_Trimestral', 'PIB_El_Salvador_Anual', 'Inflacion_El_Salvador_Anual', 'Deportaciones_Total', 'Remesas_Lag1', 'Remesas_Lag2', 'Remesas_Lag3', 'Remesas_Lag12', 'Dummy_Crisis', 'Dummy_Covid']

Distribución de las variables dummy:
Dummy_Crisis    48
Dummy_Covid     36
dtype: int64

Últimos 50 registros:
            Dummy_Crisis  Dummy_Covid
Fecha                                
2021-11-01             0            1
2021-12-01             0            1
2022-01-01             0            1
2022-02-01             0            1
2022-03-01             0            1
2022-04-01             0            1
2022-05-01             0            1
2022-06-01             0            1
2022-07-01             0            1
2022-08-01             0            1
2022-09-01             0            1
2022-10-01             0            1
2022-11-01           

In [14]:
# ============================================================
# 13. AUDITORÍA ESTRUCTURAL PREVIA A LA EXPORTACIÓN
# ============================================================

print("=== AUDITORÍA ESTRUCTURAL PREVIA A LA EXPORTACIÓN ===")

# Dimensiones.
print(f"\nDimensiones: {df_maestro_final.shape}")

# Cobertura temporal.
print(
    "\nPeríodo: "
    f"{df_maestro_final.index.min().date()} "
    f"→ {df_maestro_final.index.max().date()}"
)

# Valores faltantes.
print("\nNulos por variable:")
print(df_maestro_final.isnull().sum())

# Tipos de datos.
print("\nTipos de datos:")
print(df_maestro_final.dtypes)

# Variables disponibles.
print("\nVariables:")
for i, variable in enumerate(
    df_maestro_final.columns,
    start=1
):
    print(f"{i:02d}. {variable}")

# Verificaciones estructurales.
print("\n=== VERIFICACIONES ESTRUCTURALES ===")

print(
    f"Índice temporal ordenado: "
    f"{df_maestro_final.index.is_monotonic_increasing}"
)

print(
    f"Fechas duplicadas: "
    f"{df_maestro_final.index.duplicated().sum()}"
)

print(
    f"Total de observaciones: "
    f"{len(df_maestro_final)}"
)

print(
    f"Total de variables: "
    f"{df_maestro_final.shape[1]}"
)

=== AUDITORÍA ESTRUCTURAL PREVIA A LA EXPORTACIÓN ===

Dimensiones: (420, 15)

Período: 1991-01-01 → 2025-12-01

Nulos por variable:
Remesas_Millones_USD            0
Desempleo_Hispano_US            0
Desempleo_General_US            0
Inflacion_CPI_US                0
Salario_Promedio_US             0
PIB_US_Trimestral               0
PIB_El_Salvador_Anual           0
Inflacion_El_Salvador_Anual     0
Deportaciones_Total            48
Remesas_Lag1                    1
Remesas_Lag2                    2
Remesas_Lag3                    3
Remesas_Lag12                  12
Dummy_Crisis                    0
Dummy_Covid                     0
dtype: int64

Tipos de datos:
Remesas_Millones_USD           float64
Desempleo_Hispano_US           float64
Desempleo_General_US           float64
Inflacion_CPI_US               float64
Salario_Promedio_US            float64
PIB_US_Trimestral              float64
PIB_El_Salvador_Anual          float64
Inflacion_El_Salvador_Anual    float64
Deportaciones_T

In [15]:
# ============================================================
# 14. ESTANDARIZACIÓN Y EXPORTACIÓN DEL DATASET MAESTRO
# ============================================================
# NOTA DE REPRODUCIBILIDAD:
# Algunas fuentes externas consultadas mediante API, como FRED y
# Banco Mundial, pueden actualizar o revisar sus series históricas.
# Por esta razón, una nueva ejecución del pipeline puede producir
# diferencias puntuales respecto al dataset utilizado originalmente
# para los resultados reportados en el informe.
# Estandarizar los nombres finales de las variables económicas.

df_maestro_final = df_maestro_final.rename(
    columns={
        'PIB_US_Trimestral': 'PIB_US',
        'PIB_El_Salvador_Anual': 'PIB_ES',
        'Inflacion_El_Salvador_Anual': 'Inflacion_ES'
    }
)

# Definir el orden final de las variables.
columnas_finales = [
    'Remesas_Millones_USD',
    'Remesas_Lag1',
    'Remesas_Lag2',
    'Remesas_Lag3',
    'Remesas_Lag12',
    'Desempleo_Hispano_US',
    'Desempleo_General_US',
    'Salario_Promedio_US',
    'Inflacion_CPI_US',
    'PIB_US',
    'Inflacion_ES',
    'PIB_ES',
    'Deportaciones_Total',
    'Dummy_Crisis',
    'Dummy_Covid'
]

# Aplicar el orden definido.
df_maestro_final = df_maestro_final[columnas_finales]

# Definir la ruta de salida del dataset maestro.
ruta_salida = DATA_DIR / 'Dataset_Maestro_Final.csv'

df_maestro_final.to_csv(
    ruta_salida,
    encoding='utf-8-sig'
)

# Verificación de la exportación.
print("=== EXPORTACIÓN DEL DATASET MAESTRO ===")
print(f"\nArchivo generado: {ruta_salida}")
print(f"Dimensiones: {df_maestro_final.shape}")

print("\nValores nulos por variable:")
print(df_maestro_final.isnull().sum())

print("\nPrimeros 24 registros:")
print(df_maestro_final.head(24))

=== EXPORTACIÓN DEL DATASET MAESTRO ===

Archivo generado: C:\Users\kmabe\TFM\data\Dataset_Maestro_Final.csv
Dimensiones: (420, 15)

Valores nulos por variable:
Remesas_Millones_USD     0
Remesas_Lag1             1
Remesas_Lag2             2
Remesas_Lag3             3
Remesas_Lag12           12
Desempleo_Hispano_US     0
Desempleo_General_US     0
Salario_Promedio_US      0
Inflacion_CPI_US         0
PIB_US                   0
Inflacion_ES             0
PIB_ES                   0
Deportaciones_Total     48
Dummy_Crisis             0
Dummy_Covid              0
dtype: int64

Primeros 24 registros:
            Remesas_Millones_USD  Remesas_Lag1  Remesas_Lag2  Remesas_Lag3  \
Fecha                                                                        
1991-01-01                  63.1           NaN           NaN           NaN   
1991-02-01                  58.4          63.1           NaN           NaN   
1991-03-01                  67.6          58.4          63.1           NaN   
1991-04